In [1]:
import pandas as pd
import json
from datetime import datetime
pd.set_option('display.max_columns', None)

_EXTRACTED_META_COLS = ["_filter_param", "_filter_value", "_extract_datetime"]

def _add_openalex_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df

def _add_openalex_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = datetime.today()
    df["_load_datetime"] = pd.to_datetime(load_datetime)
    return df


In [2]:
df_work_raw = catalog.load('raw/openalex/work_dev#parquet')

[03/04/26 11:04:42] INFO     Loading data from raw/openalex/work_dev#parquet                   ]8;id=841321;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=990152;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

# Nodo

In [3]:
def openalex_load_work_concept(df_work_raw):
    df_work_raw = _add_openalex_extracted_metadata(df_work_raw)

    df_work = df_work_raw.loc[:, ['id', 'concepts', '_filter_param', '_filter_value', '_extract_datetime']]
    df_work = df_work.convert_dtypes()

    df_work2concepts_exploded = df_work.explode('concepts').reset_index(drop=True)
    df_work2concepts_norm = pd.json_normalize(df_work2concepts_exploded['concepts'])
    df_work2concepts_norm.rename(columns={'id':'concept_id'}, inplace=True)

    df_work2concepts = pd.concat(
        (df_work2concepts_exploded.loc[:, ['id', '_filter_param', '_filter_value', '_extract_datetime']], df_work2concepts_norm),
        axis=1,
    )
    
    df_work2concepts = _add_openalex_loaded_metadata(df_work2concepts)

    return df_work2concepts


## Ejecuto Nodo

In [4]:
df_work2concepts = openalex_load_work_concept(df_work_raw)

# Resultados

In [5]:
df_work2concepts

,id,_filter_param,_filter_value,_extract_datetime,display_name,concept_id,level,score,wikidata,_load_datetime
0,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Climate change,https://openalex.org/C132651083,2,0.814506,https://www.wikidata.org/wiki/Q7942,2026-03-04 11:04:44.298217
1,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Global warming,https://openalex.org/C115343472,3,0.555365,https://www.wikidata.org/wiki/Q7942,2026-03-04 11:04:44.298217
2,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Ecosystem,https://openalex.org/C110872660,2,0.554159,https://www.wikidata.org/wiki/Q37813,2026-03-04 11:04:44.298217
3,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Ecology,https://openalex.org/C18903297,1,0.519316,https://www.wikidata.org/wiki/Q7150,2026-03-04 11:04:44.298217
4,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Global change,https://openalex.org/C199491958,3,0.489902,https://www.wikidata.org/wiki/Q737514,2026-03-04 11:04:44.298217
...,...,...,...,...,...,...,...,...,...,...
2972,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Engineering,https://openalex.org/C127413603,0,0.222143,https://www.wikidata.org/wiki/Q11023,2026-03-04 11:04:44.298217
2973,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Materials science,https://openalex.org/C192562407,0,0.210661,https://www.wikidata.org/wiki/Q228736,2026-03-04 11:04:44.298217
2974,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Biotechnology,https://openalex.org/C150903083,1,0.169365,https://www.wikidata.org/wiki/Q7108,2026-03-04 11:04:44.298217
2975,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Biology,https://openalex.org/C86803240,0,0.089647,https://www.wikidata.org/wiki/Q420,2026-03-04 11:04:44.298217
